In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers
!pip install -q scikit-learn

In [ ]:
# ====== Core ======
import os
import json
import random
from typing import List, Dict

# ====== Torch ======
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ====== Transformers ======
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW


# ====== Mixed Precision ======
from torch.amp import autocast, GradScaler

# ====== Metrics ======
from sklearn.metrics import precision_recall_fscore_support

In [ ]:
# ====== Reproducibility ======
SEED = 42
TRAIN_RATIO = 0.9
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ====== Device ======
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ====== Model ======
MODEL_NAME = "xlm-roberta-base"

# ====== Training ======
MAX_LEN = 128
BATCH_SIZE = 10
GRAD_ACCUM_STEPS = 2
EPOCHS = 6
FP16 = True

# ====== Optimization ======
LR_ENCODER = 2e-5
LR_CLASSIFIER = 1e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# ====== Loss ======
IGNORE_INDEX = -100
LAMBDA_PARALLEL = 0.1

# ===== CONFIG =====
BASE_DIR = "/content/drive/MyDrive/en_vi_sense_tagger"
DATA_DIR = os.path.join(BASE_DIR, "data")
MODEL_DIR = os.path.join(BASE_DIR, "model")


INPUT_FILE = os.path.join(DATA_DIR, "dataset.jsonl")
TRAIN_FILE = os.path.join(DATA_DIR, "dataset_train.jsonl")
DEV_FILE   = os.path.join(DATA_DIR, "dataset_dev.jsonl")

In [ ]:
def load_label_map(en_sense_path, vi_sense_path):
    def extract_sense_ids(json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        sense_ids = []

        for lemma, pos_dict in data.items():
            for pos, senses in pos_dict.items():
                for sense in senses:
                    sense_id = sense["id"]
                    sense_ids.append(sense_id)

        return sorted(list(set(sense_ids)))

    en_sense_ids = extract_sense_ids(en_sense_path)
    vi_sense_ids = extract_sense_ids(vi_sense_path)

    EN_LABELS = ["PAD", "O"] + en_sense_ids
    VI_LABELS = ["PAD", "O"] + vi_sense_ids

    EN_LABEL2ID = {label: idx for idx, label in enumerate(EN_LABELS)}
    VI_LABEL2ID = {label: idx for idx, label in enumerate(VI_LABELS)}

    EN_ID2LABEL = {v: k for k, v in EN_LABEL2ID.items()}
    VI_ID2LABEL = {v: k for k, v in VI_LABEL2ID.items()}

    return EN_LABEL2ID, EN_ID2LABEL, VI_LABEL2ID, VI_ID2LABEL

In [ ]:
class ParallelSenseDataset(Dataset):
    """
    Cached Dataset cho English–Vietnamese Parallel Sense Tagging.

    Mỗi dòng jsonl:
    {
      "en": {"tokens": [...], "labels": [...]},
      "vi": {"tokens": [...], "labels": [...]}
    }
    """

    def __init__(
        self,
        jsonl_path: str,
        tokenizer,
        en_label2id: dict,
        vi_label2id: dict,
        max_len: int = 128,
        cache: bool = True
    ):
        self.tokenizer = tokenizer
        self.en_label2id = en_label2id
        self.vi_label2id = vi_label2id
        self.max_len = max_len
        self.cache = cache

        # ---- stats ----
        self.unknown_sense_counter = {"en": {}, "vi": {}}
        self.skipped = 0

        # ---- raw samples ----
        raw_samples = []

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    sample = json.loads(line)

                    # strict schema check
                    if (
                        "en" not in sample or "vi" not in sample or
                        "tokens" not in sample["en"] or
                        "labels" not in sample["en"] or
                        "tokens" not in sample["vi"] or
                        "labels" not in sample["vi"]
                    ):
                        self.skipped += 1
                        continue

                    if (
                        len(sample["en"]["tokens"]) != len(sample["en"]["labels"]) or
                        len(sample["vi"]["tokens"]) != len(sample["vi"]["labels"])
                    ):
                        self.skipped += 1
                        continue

                    raw_samples.append(sample)

                except Exception:
                    self.skipped += 1

        print(f"✓ Loaded {len(raw_samples)} valid samples")
        if self.skipped > 0:
            print(f"⚠️ Skipped {self.skipped} invalid samples")

        # ---- CACHE ----
        if self.cache:
            print("⏳ Caching tokenization & label alignment...")
            self.samples = [self._encode_pair(s) for s in raw_samples]
            print("✓ Caching completed")
        else:
            self.samples = raw_samples

    def _encode_one(self, tokens, labels, label2id, lang):
        input_ids = []
        label_ids = []

        for tok, lab in zip(tokens, labels):
            sub_tokens = self.tokenizer.tokenize(tok)

            for i, st in enumerate(sub_tokens):
                input_ids.append(self.tokenizer.convert_tokens_to_ids(st))

                if i == 0:
                    if lab in label2id:
                        label_ids.append(label2id[lab])
                    else:
                        # OOV sense → map to O
                        label_ids.append(label2id["O"])
                        self.unknown_sense_counter[lang][lab] = (
                            self.unknown_sense_counter[lang].get(lab, 0) + 1
                        )
                else:
                    label_ids.append(IGNORE_INDEX)

        attention_mask = [1] * len(input_ids)

        # pad / truncate
        if len(input_ids) < self.max_len:
            pad_len = self.max_len - len(input_ids)
            input_ids += [self.tokenizer.pad_token_id] * pad_len
            attention_mask += [0] * pad_len
            label_ids += [IGNORE_INDEX] * pad_len
        else:
            input_ids = input_ids[:self.max_len]
            attention_mask = attention_mask[:self.max_len]
            label_ids = label_ids[:self.max_len]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }

    def _encode_pair(self, sample):
        return {
            "en": self._encode_one(
                sample["en"]["tokens"],
                sample["en"]["labels"],
                self.en_label2id,
                lang="en"
            ),
            "vi": self._encode_one(
                sample["vi"]["tokens"],
                sample["vi"]["labels"],
                self.vi_label2id,
                lang="vi"
            )
        }

    def __getitem__(self, idx):
        return self.samples[idx]

    def __len__(self):
        return len(self.samples)

In [ ]:
class XLMRParallelSenseTagger(nn.Module):
    def __init__(self, model_name, num_en_labels, num_vi_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.en_classifier = nn.Linear(hidden_size, num_en_labels)
        self.vi_classifier = nn.Linear(hidden_size, num_vi_labels)

    def forward(self, en_batch, vi_batch):
        en_out = self.encoder(
            input_ids=en_batch["input_ids"],
            attention_mask=en_batch["attention_mask"]
        )

        vi_out = self.encoder(
            input_ids=vi_batch["input_ids"],
            attention_mask=vi_batch["attention_mask"]
        )

        en_hidden = self.dropout(en_out.last_hidden_state)
        vi_hidden = self.dropout(vi_out.last_hidden_state)

        en_logits = self.en_classifier(en_hidden)
        vi_logits = self.vi_classifier(vi_hidden)

        return en_logits, vi_logits, en_hidden, vi_hidden

In [ ]:
def supervised_loss(logits, labels):
    loss_fct = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
    return loss_fct(
        logits.view(-1, logits.size(-1)),
        labels.view(-1)
    )

In [ ]:
def parallel_consistency_loss(en_hidden, en_labels,
                              vi_hidden, vi_labels):
    mask = (
        (en_labels != IGNORE_INDEX) &
        (vi_labels != IGNORE_INDEX) &
        (en_labels != 1) &   # O label
        (vi_labels != 1)
    )

    if mask.sum() == 0:
        return torch.tensor(0.0, device=en_hidden.device)

    en_vecs = en_hidden[mask]
    vi_vecs = vi_hidden[mask]

    cosine_sim = nn.functional.cosine_similarity(en_vecs, vi_vecs, dim=-1)
    return torch.mean(1 - cosine_sim)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, scaler, epoch):
    model.train()
    total_loss = 0.0

    optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        en_batch = {k: v.to(DEVICE) for k, v in batch["en"].items()}
        vi_batch = {k: v.to(DEVICE) for k, v in batch["vi"].items()}

        with autocast(device_type="cuda", enabled=FP16):
            en_logits, vi_logits, en_hidden, vi_hidden = model(en_batch, vi_batch)

            loss_en = supervised_loss(en_logits, en_batch["labels"])
            loss_vi = supervised_loss(vi_logits, vi_batch["labels"])
            loss_parallel = parallel_consistency_loss(
                en_hidden, en_batch["labels"],
                vi_hidden, vi_batch["labels"]
            )

            loss = loss_en + loss_vi + LAMBDA_PARALLEL * loss_parallel
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        total_loss += loss.item()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

    return total_loss

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ===== LOAD DATA =====
samples = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        samples.append(line.strip())

print(f"Total samples: {len(samples)}")

# ===== SHUFFLE =====
random.shuffle(samples)

# ===== SPLIT =====
split_idx = int(len(samples) * TRAIN_RATIO)
train_samples = samples[:split_idx]
dev_samples = samples[split_idx:]

print(f"Train samples: {len(train_samples)}")
print(f"Dev samples:   {len(dev_samples)}")

# ===== SAVE =====
with open(TRAIN_FILE, "w", encoding="utf-8") as f:
    for line in train_samples:
        f.write(line + "\n")

with open(DEV_FILE, "w", encoding="utf-8") as f:
    for line in dev_samples:
        f.write(line + "\n")

print("✓ Dataset split completed")
print(f"→ {TRAIN_FILE}")
print(f"→ {DEV_FILE}")

Total samples: 49484
Train samples: 44535
Dev samples:   4949
✓ Dataset split completed
→ /content/drive/MyDrive/en_vi_sense_tagger/data/dataset_train.jsonl
→ /content/drive/MyDrive/en_vi_sense_tagger/data/dataset_dev.jsonl


In [ ]:
def evaluate_dev(model, dataloader, en_id2label, vi_id2label):
    model.eval()

    en_true, en_pred = [], []
    vi_true, vi_pred = [], []

    with torch.no_grad():
        for batch in dataloader:
            en_batch = {k: v.to(DEVICE) for k, v in batch["en"].items()}
            vi_batch = {k: v.to(DEVICE) for k, v in batch["vi"].items()}

            en_logits, vi_logits, _, _ = model(en_batch, vi_batch)

            en_preds = torch.argmax(en_logits, dim=-1)
            vi_preds = torch.argmax(vi_logits, dim=-1)

            # EN
            for p, l in zip(en_preds.view(-1), en_batch["labels"].view(-1)):
                if l.item() != IGNORE_INDEX:
                    en_true.append(l.item())
                    en_pred.append(p.item())

            # VI
            for p, l in zip(vi_preds.view(-1), vi_batch["labels"].view(-1)):
                if l.item() != IGNORE_INDEX:
                    vi_true.append(l.item())
                    vi_pred.append(p.item())

    en_p, en_r, en_f1, _ = precision_recall_fscore_support(
        en_true, en_pred, average="micro", zero_division=0
    )

    vi_p, vi_r, vi_f1, _ = precision_recall_fscore_support(
        vi_true, vi_pred, average="micro", zero_division=0
    )

    return {
        "en_f1": en_f1,
        "vi_f1": vi_f1,
        "avg_f1": (en_f1 + vi_f1) / 2
    }

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Label maps
EN_LABEL2ID, EN_ID2LABEL, VI_LABEL2ID, VI_ID2LABEL = load_label_map(
    os.path.join(DATA_DIR, "en_sense_list.json"),
    os.path.join(DATA_DIR, "vi_sense_list.json")
)

# Datasets
train_dataset = ParallelSenseDataset(
    os.path.join(DATA_DIR, "dataset_train.jsonl"),
    tokenizer,
    EN_LABEL2ID,
    VI_LABEL2ID,
    MAX_LEN
)

dev_dataset = ParallelSenseDataset(
    os.path.join(DATA_DIR, "dataset_dev.jsonl"),
    tokenizer,
    EN_LABEL2ID,
    VI_LABEL2ID,
    MAX_LEN
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# Model
model = XLMRParallelSenseTagger(
    MODEL_NAME,
    num_en_labels=len(EN_LABEL2ID),
    num_vi_labels=len(VI_LABEL2ID)
).to(DEVICE)

# Optimizer (phân tầng LR)
optimizer = AdamW(
    [
        {"params": model.encoder.parameters(), "lr": LR_ENCODER},
        {"params": model.en_classifier.parameters(), "lr": LR_CLASSIFIER},
        {"params": model.vi_classifier.parameters(), "lr": LR_CLASSIFIER},
    ],
    weight_decay=WEIGHT_DECAY
)

scaler = GradScaler(enabled=FP16)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✓ Loaded 44450 valid samples
⚠️ Skipped 85 invalid samples
⏳ Caching tokenization & label alignment...
✓ Caching completed
✓ Loaded 4938 valid samples
⚠️ Skipped 11 invalid samples
⏳ Caching tokenization & label alignment...
✓ Caching completed


In [ ]:
best_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        epoch
    )

    metrics = evaluate_dev(
        model,
        dev_loader,
        EN_ID2LABEL,
        VI_ID2LABEL
    )

    print(
        f"Train loss: {train_loss:.4f} | "
        f"EN F1: {metrics['en_f1']:.4f} | "
        f"VI F1: {metrics['vi_f1']:.4f} | "
        f"AVG F1: {metrics['avg_f1']:.4f}"
    )

    # Save best
    if metrics["avg_f1"] > best_f1:
        best_f1 = metrics["avg_f1"]

        save_path = os.path.join(MODEL_DIR, "best_model.pt")
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "en_label2id": EN_LABEL2ID,
                "vi_label2id": VI_LABEL2ID,
            },
            save_path
        )

        print(f"✓ Best model saved to {save_path}")


===== Epoch 1/6 =====
Train loss: 2946.6301 | EN F1: 0.9630 | VI F1: 0.8881 | AVG F1: 0.9255
✓ Best model saved to /content/drive/MyDrive/en_vi_sense_tagger/model/best_model.pt

===== Epoch 2/6 =====
Train loss: 1530.0552 | EN F1: 0.9667 | VI F1: 0.8933 | AVG F1: 0.9300
✓ Best model saved to /content/drive/MyDrive/en_vi_sense_tagger/model/best_model.pt

===== Epoch 3/6 =====
Train loss: 1316.5739 | EN F1: 0.9676 | VI F1: 0.8951 | AVG F1: 0.9313
✓ Best model saved to /content/drive/MyDrive/en_vi_sense_tagger/model/best_model.pt

===== Epoch 4/6 =====
Train loss: 1181.0566 | EN F1: 0.9690 | VI F1: 0.8981 | AVG F1: 0.9336
✓ Best model saved to /content/drive/MyDrive/en_vi_sense_tagger/model/best_model.pt

===== Epoch 5/6 =====
Train loss: 1085.9932 | EN F1: 0.9691 | VI F1: 0.8974 | AVG F1: 0.9332

===== Epoch 6/6 =====
Train loss: 999.0983 | EN F1: 0.9694 | VI F1: 0.8969 | AVG F1: 0.9332
